# db_09: Neuaufbau aus den Excel-Dateien

**Frage:** Kann XGBoost aus den Excel-Daten (Argus + Marktdaten) die 3-Tage-Richtung des Heizöl- und Dieselpreises besser vorhersagen als Zufall und als einfache Regeln?

**Ablauf (Zelle für Zelle ausführen, nach jeder Zelle die Ausgabe lesen):**
1. Setup und Pfade
2. Struktur der Marktdaten-Excel ansehen
3. Daten laden und prüfen
4. Abgleich mit der Gold-Tabelle (zeigt, ob der Neuaufbau identisch ist)
5. Leakage-Test
6. Schnelltest: eine Serie
7. Alle 8 Serien (ca. 15 Minuten)
8. Auswertung mit Holm-Korrektur

Der Code liegt im Repo-Ordner `p229/` und ist mit `pytest` getestet. Keine Zugangsdaten nötig.

In [ ]:
%pip install xgboost scikit-learn openpyxl

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Zelle 0: Legt den Projektcode (Ordner p229) auf dem Cluster an. Nichts ändern, nur ausführen.
import os
os.makedirs("/tmp/p229_code/p229", exist_ok=True)
FILES = {}
FILES['__init__'] = r'''"""P229: Kauf/Warten-Signal für Heizöl und Diesel (BayWa). Logik aus db_08_xgboost_sauber."""
'''
FILES['config'] = r'''"""Konstanten aus db_08 (unverändert übernommen)."""
import pandas as pd

SERIES = ["heizoel_sued", "heizoel_suedwest", "heizoel_suedost", "heizoel_rheinmain",
          "diesel_sued", "diesel_suedwest", "diesel_suedost", "diesel_rheinmain"]
EXO = ["ice_gasoil", "brent", "wti", "nymex_heating_oil", "usd_eur"]
LAST_ARGUS = pd.Timestamp("2026-01-20")   # danach OMR + fortgeschriebener ICE-Preis
DEV = (pd.Timestamp("2023-04-21"), pd.Timestamp("2024-11-01"))
TEST = (pd.Timestamp("2024-11-15"), pd.Timestamp("2026-01-20"))
HORIZON = 3
REG = dict(max_depth=2, n_estimators=150, learning_rate=0.03, min_child_weight=10,
           reg_lambda=10, subsample=0.7, colsample_bytree=0.5,
           eval_metric="logloss", random_state=42)
# Szenario -> (entry_lag, feat_lag)
SCEN = {"S0 (Kauf zu Preis t)": (0, 0),
        "S1 (Kauf zu Preis t+1)": (1, 0),
        "S2 (Daten nur bis t-1)": (0, 1)}
'''
FILES['stats'] = r'''from math import sqrt

import numpy as np
from scipy.stats import binomtest


def wil(k, n, z=1.96):
    """Wilson-95-%-Intervall für eine Trefferquote k/n."""
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h


def block_boot_mean(x, block=5, n_boot=2000, seed=42):
    """95-%-Intervall des Mittelwerts per Block-Bootstrap (berücksichtigt überlappende Labels)."""
    x = np.asarray(x, float)
    n = len(x)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(n / block))
    starts = np.arange(n - block + 1)
    means = np.empty(n_boot)
    for b in range(n_boot):
        s = rng.choice(starts, nb)
        means[b] = x[(s[:, None] + np.arange(block)).ravel()[:n]].mean()
    return np.percentile(means, [2.5, 97.5])


def mcnemar(model_ok, other_ok):
    """Exakter McNemar-Test (gepaarte Treffer, zweiseitig)."""
    oa, ob = int((model_ok & ~other_ok).sum()), int((~model_ok & other_ok).sum())
    return binomtest(oa, oa + ob, 0.5).pvalue if oa + ob else 1.0


def holm(p):
    """Holm-Bonferroni-adjustierte p-Werte."""
    p = np.asarray(p, float)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[idx])
        adj[idx] = min(1.0, run)
    return adj
'''
FILES['features'] = r'''import numpy as np
import pandas as pd

from .config import EXO


def build_features(df, H, exo=EXO):
    """Stationäre Merkmale (Log-Returns, Nachlauf-Lücke, Volatilität, Wochentag) für Serie H.

    Jedes Merkmal in Zeile t nutzt nur Preise bis einschließlich t.
    """
    df = df.reset_index(drop=True)
    F = pd.DataFrame({"date": df["date"]})
    lp = {c: np.log(df[c].where(df[c] > 0)) for c in [H] + list(exo)}
    for c, s in lp.items():
        r1 = s.diff()
        F[f"{c}_r1"] = r1
        F[f"{c}_r1_l1"] = r1.shift(1)
        F[f"{c}_r1_l2"] = r1.shift(2)
        F[f"{c}_r3"] = s - s.shift(3)
        F[f"{c}_r5"] = s - s.shift(5)
    for c in exo:                                   # Nachlauf-Lücke: Marktbewegung minus Preisbewegung
        F[f"gap1_{c}"] = F[f"{c}_r1"] - F[f"{H}_r1"]
        F[f"gap3_{c}"] = F[f"{c}_r3"] - F[f"{H}_r3"]
    r1h = F[f"{H}_r1"]
    F["vol5_H"] = r1h.rolling(5).std()
    F["vol20_H"] = r1h.rolling(20).std()
    F["dev20_H"] = lp[H] - lp[H].rolling(20).mean()
    F["weekday"] = df["date"].dt.weekday
    return F


def london_features(H, F):
    """Argus-only-Variante: eigene Serie + ICE Gasoil + EUR/USD."""
    keep = ["date", "weekday", "vol5_H", "vol20_H", "dev20_H"]
    for c in [H, "ice_gasoil", "usd_eur"]:
        keep += [f"{c}_r1", f"{c}_r1_l1", f"{c}_r1_l2", f"{c}_r3", f"{c}_r5"]
    for c in ["ice_gasoil", "usd_eur"]:
        keep += [f"gap1_{c}", f"gap3_{c}"]
    return F[keep]
'''
FILES['leakage'] = r'''import numpy as np

from .features import build_features


def leak_test(d, H, n_cut=1500, n_check=50):
    """Features dürfen sich nicht ändern, wenn man die Zukunft abschneidet.

    Braucht mindestens n_cut Zeilen. Wirft AssertionError bei Look-ahead.
    """
    if len(d) < n_cut:
        raise ValueError(f"leak_test braucht >= {n_cut} Zeilen, d hat {len(d)}")
    a = build_features(d.iloc[:n_cut], H).iloc[n_cut - n_check:n_cut].drop(columns="date").values
    b = build_features(d, H).iloc[n_cut - n_check:n_cut].drop(columns="date").values
    assert np.allclose(a, b, equal_nan=True), f"Look-ahead in den Features ({H})!"
'''
FILES['walk_forward'] = r'''import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from .config import HORIZON, REG


def walk_forward(d, H, F, window, entry_lag=0, feat_lag=0, retrain_every=5, params=REG, horizon=HORIZON):
    """Gepurgter Walk-Forward (expanding window).

    d: Datentabelle (Spalte date + Preisspalten), F: Features zu d, window: (start, ende).
    Ziel: Preis(t+horizon) > Preis(t+entry_lag). Features: Information bis t-feat_lag.
    Training nur mit Zeilen, deren Label am Entscheidungstag i schon bekannt ist:
    last = i - horizon - feat_lag.
    """
    feats = [c for c in F.columns if c != "date"]
    P = d[H].astype(float)
    p0, p1 = P.shift(-entry_lag), P.shift(-horizon)
    y = (p1 > p0).astype(float).where(p0.notna() & p1.notna())
    X = F[feats].shift(feat_lag).values.astype(float)
    yv = y.values
    idx = np.where((d["date"] >= window[0]) & (d["date"] <= window[1]))[0]
    proba, model = [], None
    for k, i in enumerate(idx):
        if model is None or k % retrain_every == 0:
            last = i - horizon - feat_lag
            Xt, yt = X[:last + 1], yv[:last + 1]
            m = ~np.isnan(yt)
            model = XGBClassifier(**params).fit(Xt[m], yt[m])
        proba.append(model.predict_proba(X[i:i + 1])[0, 1])
    res = pd.DataFrame({"date": d["date"].values[idx], "y_true": yv[idx], "proba": proba})
    return res.dropna(subset=["y_true"]).reset_index(drop=True)
'''
FILES['evaluate'] = r'''import numpy as np
import pandas as pd
from scipy.stats import binomtest

from .config import HORIZON
from .stats import block_boot_mean, mcnemar, wil


def evaluate_run(d, H, F, res, entry_lag, feat_lag, horizon=HORIZON):
    """Pflichtmetriken für einen Walk-Forward-Lauf (Logik 1:1 aus db_08).

    F muss das volle Merkmalsset enthalten (für die Baselines NYMEX-Regel und Momentum).
    """
    y, p = res["y_true"].values, res["proba"].values
    pred = (p >= 0.5).astype(int)
    ok = (pred == y)
    n = len(res)
    hit = int(ok.sum())
    lo, hi = wil(hit, n)
    blo, bhi = block_boot_mean(ok.astype(float))
    p_non = max(binomtest(int(ok[o::3].sum()), len(ok[o::3]), 0.5).pvalue for o in range(3))
    base = y.mean()
    brier, ref = np.mean((p - y) ** 2), np.mean((base - y) ** 2)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    # Vorzeichenregeln als Baselines (gleiche Tage)
    pos = pd.Series(np.arange(len(d)), index=d["date"])
    i = pos.reindex(res["date"]).values.astype(int)
    nym = (F["nymex_heating_oil_r1"].shift(feat_lag).values[i] > 0).astype(int)
    mom = (F[f"{H}_r1"].shift(feat_lag).values[i] > 0).astype(int)
    # Kosten (Regret): kaufen = Preis(t+entry_lag), warten = Preis(t+horizon)
    p_now, p_fut = d[H].values[i + entry_lag], d[H].values[i + horizon]
    best = np.minimum(p_now, p_fut)
    c_mod = np.where(pred == 1, p_now, p_fut) - best
    c_buy = p_now - best
    diff = c_buy - c_mod                                   # > 0: Modell billiger als immer kaufen
    dlo, dhi = block_boot_mean(diff)
    return {"Serie": H, "n": n, "DA": hit / n, "Wilson_lo": lo, "Wilson_hi": hi,
            "Boot_lo": blo, "Boot_hi": bhi, "p_nicht_ueberl": p_non,
            "Mehrheit": max(base, 1 - base), "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1),
            "Brier_Skill": 1 - brier / ref,
            "DA_NYMEX_Regel": float((nym == y).mean()), "DA_Momentum": float((mom == y).mean()),
            "p_McNemar_NYMEX": mcnemar(ok, nym == y), "p_McNemar_Momentum": mcnemar(ok, mom == y),
            "Regret_Modell": c_mod.mean(), "Regret_immer_kaufen": c_buy.mean(),
            "Ersparnis_vs_kaufen": diff.mean(), "Ersp_lo": dlo, "Ersp_hi": dhi,
            "davon_zu_frueh_gekauft": c_mod[pred == 1].sum() / n,
            "davon_zu_lange_gewartet": c_mod[pred == 0].sum() / n}
'''
FILES['data'] = r'''"""Daten aus den Excel-Dateien laden und zu einer Tabelle `d` zusammenführen.

Argus-Excel (Add-in): Blatt "Tabelle1", Zeile 2 = Beschreibung, ab Zeile 3 Tageswerte.
Marktdaten-Excel: je Blatt Datum + Wert (Brent, WTI, Heizöl/NYMEX, Wechselkurs).
"""
import warnings

import numpy as np
import pandas as pd

from .config import EXO, SERIES

_REGION = [("southwest", "suedwest"), ("southeast", "suedost"), ("rhine-main", "rheinmain"), ("south", "sued")]


def _serie_name(desc):
    """Beschreibungstext der Argus-Spalte -> Serienname (oder None, z. B. 'all regions')."""
    t = str(desc).lower()
    if "ice nwe month 1" in t:
        return "ice_gasoil"
    if "all regions" in t:
        return None
    produkt = "diesel" if "diesel en 590" in t else "heizoel" if "heating oil" in t else None
    if produkt is None:
        return None
    for key, name in _REGION:
        if key in t:
            return f"{produkt}_{name}"
    return None


def load_argus_excel(path, sheet="Tabelle1"):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    head = raw.iloc[1]                                   # Zeile 2: Beschreibungen
    body = raw.iloc[2:].copy()
    cols = {0: "date"}
    for j in range(1, raw.shape[1]):
        n = _serie_name(head.iloc[j])
        if n:
            cols[j] = n
    df = body[list(cols)].rename(columns=cols)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date")
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    fehlt = [c for c in SERIES + ["ice_gasoil"] if c not in df.columns]
    if fehlt:
        raise ValueError(f"Argus-Spalten nicht erkannt: {fehlt}")
    return df.drop_duplicates("date", keep="last").reset_index(drop=True)


def inspect_excel(path, n=6):
    """Struktur jeder Tabelle ausgeben (Blattnamen, Größe, erste Zeilen) – zum Prüfen vor dem Laden."""
    xl = pd.ExcelFile(path)
    for s in xl.sheet_names:
        df = xl.parse(s, header=None, nrows=n + 4)
        print(f"--- Blatt '{s}' (erste Zeilen) ---")
        print(df.head(n + 4).to_string(max_colwidth=40))


def load_market_sheet(path, sheet, name, date_col=None, value_col=None, header_row=None):
    """Ein Marktdaten-Blatt als Serie `name`.

    Ohne Angabe wird das Blatt automatisch gelesen: erste Spalte mit Datumswerten, danach die
    erste Spalte mit überwiegend Zahlen. Bei Zweifel date_col/value_col (0-basiert) und
    header_row angeben, nachdem `inspect_excel` die Struktur gezeigt hat.
    """
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dt = raw.apply(lambda s: pd.to_datetime(s, errors="coerce"))
    if date_col is None:
        is_date = [(dt[c].notna() & pd.to_numeric(raw[c], errors="coerce").isna()).mean() for c in raw.columns]
        date_col = int(np.argmax(is_date))
        if max(is_date) < 0.5:
            raise ValueError(f"Blatt '{sheet}': keine Datumsspalte gefunden")
    if value_col is None:
        num = [(pd.to_numeric(raw[c], errors="coerce").notna().mean() if c != date_col else -1) for c in raw.columns]
        value_col = int(np.argmax(num))
    out = pd.DataFrame({"date": dt[date_col], name: pd.to_numeric(raw[value_col], errors="coerce")})
    out = out.dropna(subset=["date", name])
    out = out[out["date"] >= "1990-01-01"].drop_duplicates("date", keep="last").sort_values("date")
    return out.reset_index(drop=True)


def build_dataset(argus, markt, max_ffill=5):
    """Argus-Handelstage als Gerüst, Marktdaten per Datum dazu, Lücken bis `max_ffill` Tage vorwärts füllen.

    Rückgabe: (d, bericht). Das Vorwärtsfüllen nutzt nur vergangene Werte (kein Look-ahead).
    """
    d = argus.copy()
    for m in markt:
        d = d.merge(m, on="date", how="left")
    cols = SERIES + EXO
    fehlt = [c for c in cols if c not in d.columns]
    if fehlt:
        raise ValueError(f"Spalten fehlen: {fehlt}")
    vorher = d[cols].isna().sum()
    d[cols] = d[cols].ffill(limit=max_ffill)
    d = d.dropna(subset=cols).reset_index(drop=True)
    bericht = pd.DataFrame({"fehlend_vor_ffill": vorher, "fehlend_nach_ffill": d[cols].isna().sum()})
    return d, bericht
'''
for name, text in FILES.items():
    open(f"/tmp/p229_code/p229/{name}.py", "w").write(text)
print("Code angelegt:", sorted(os.listdir("/tmp/p229_code/p229")))

In [ ]:
# Zelle 1: Pfade und Code finden. Nichts anpassen nötig, wenn eine der beiden Varianten unten erfüllt ist:
#  A) Das Repo ist ein Git-Ordner im Workspace und dieses Notebook liegt darin (Unterordner notebooks/).
#  B) Die Datei p229_code.zip liegt im Volume (BASE_PATH), dann wird sie automatisch entpackt.
import os, sys, time, zipfile
import numpy as np, pandas as pd

BASE_PATH = "/Volumes/dev_workspace/p229/p229_files"
OUT = f"{BASE_PATH}/output"
ARGUS_XLSX = f"{BASE_PATH}/Heizoel und Diesel bis 2016 und ICE Daten.xlsx"
MARKT_XLSX = f"{BASE_PATH}/P229 - Daten von Brent, WTI, Heizöl, Wechselkurs.xlsx"

def finde_code():
    if os.path.isdir("/tmp/p229_code/p229"):             # Variante C: Code wurde per Zelle 0 angelegt
        return "/tmp/p229_code"
    p = os.getcwd()
    for _ in range(4):                                   # Variante A: vom Notebook-Ordner nach oben suchen
        if os.path.isdir(os.path.join(p, "p229")):
            return p
        p = os.path.dirname(p)
    zip_pfad = f"{BASE_PATH}/p229_code.zip"              # Variante B: Zip im Volume
    if os.path.exists(zip_pfad):
        ziel = "/tmp/p229_code"
        zipfile.ZipFile(zip_pfad).extractall(ziel)
        return ziel
    raise FileNotFoundError("Ordner p229/ nicht gefunden. Weder Git-Ordner (A) noch p229_code.zip im Volume (B).")

CODE_PATH = finde_code()
sys.path.insert(0, CODE_PATH)
print("Code gefunden in:", CODE_PATH)

from p229.config import SERIES, EXO, DEV, TEST, SCEN, LAST_ARGUS
from p229.data import inspect_excel, load_argus_excel, load_market_sheet, build_dataset
from p229.features import build_features
from p229.leakage import leak_test
from p229.walk_forward import walk_forward
from p229.evaluate import evaluate_run
from p229.stats import holm
print("Import ok")

In [ ]:
# Zelle 2: Struktur der Marktdaten-Excel ansehen (nur lesen, nichts ändern).
# Bitte die Ausgabe kopieren und mir schicken, falls Zelle 3 meckert.
inspect_excel(MARKT_XLSX)

In [ ]:
# Zelle 3: Daten laden. Blattnamen wie im Projektdokument; falls sie anders heißen, hier anpassen.
argus = load_argus_excel(ARGUS_XLSX)
markt = [
    load_market_sheet(MARKT_XLSX, "Brent Rohöl", "brent"),
    load_market_sheet(MARKT_XLSX, "WIT Rohöl", "wti"),
    load_market_sheet(MARKT_XLSX, "Heizöl", "nymex_heating_oil"),          # ANNAHME: Blatt "Heizöl" = nymex_heating_oil
    load_market_sheet(MARKT_XLSX, "USD_EUR Wechselkurs", "usd_eur"),
]
for m in markt:
    print(m.columns[1], len(m), m["date"].min().date(), "bis", m["date"].max().date(),
          "| Min/Max:", round(m.iloc[:, 1].min(), 3), round(m.iloc[:, 1].max(), 3))
d_all, bericht = build_dataset(argus, markt)
print("\nFehlende Werte vor/nach Vorwärtsfüllen (max. 5 Tage):")
print(bericht.to_string())
d = d_all[d_all["date"] <= LAST_ARGUS].reset_index(drop=True)      # Argus-Phase wie in db_08
print("\nTabelle d:", d.shape, d["date"].min().date(), "bis", d["date"].max().date())

In [ ]:
# Zelle 4: Abgleich mit der Gold-Tabelle. Erwartung: Abweichung 0 (oder erklärbar).
g = (spark.table("dev_workspace.p229.gold_df_features").select("date", *(SERIES + EXO)).toPandas()
     .assign(date=lambda x: pd.to_datetime(x["date"])).set_index("date").sort_index())
n = d.set_index("date")
gem = n.index.intersection(g.index)
print(f"Zeilen neu: {len(n)} | Gold: {len(g[g.index <= LAST_ARGUS])} | gemeinsame Tage: {len(gem)}")
print("Tage nur neu:", len(n.index.difference(g.index)), "| nur Gold:", len(g[g.index <= LAST_ARGUS].index.difference(n.index)))
diff = (n.loc[gem, SERIES + EXO] - g.loc[gem, SERIES + EXO]).abs()
print("\nGrößte Abweichung je Spalte:")
print(diff.max().round(5).to_string())
print("\nTage mit Abweichung > 0.001:")
print((diff > 0.001).sum().to_string())

In [ ]:
# Zelle 5: Leakage-Test für alle Serien (Features dürfen die Zukunft nicht kennen).
for H in SERIES:
    leak_test(d, H)
print("Leakage-Test bestanden für", len(SERIES), "Serien")

In [ ]:
# Zelle 6: Schnelltest, eine Serie, Szenario S0, Testfenster (ca. 1-2 Minuten).
H = "heizoel_sued"
F = build_features(d, H)
el, fl = SCEN["S0 (Kauf zu Preis t)"]
t0 = time.time()
res = walk_forward(d, H, F, TEST, entry_lag=el, feat_lag=fl)
r = evaluate_run(d, H, F, res, el, fl)
print(f"Dauer {time.time()-t0:.0f}s | n={r['n']}")
print(f"Trefferquote {r['DA']:.3f} | Wilson [{r['Wilson_lo']:.3f}, {r['Wilson_hi']:.3f}] | Bootstrap [{r['Boot_lo']:.3f}, {r['Boot_hi']:.3f}]")
print(f"p (nicht überlappend, max über 3 Offsets) {r['p_nicht_ueberl']:.3f} | Mehrheitsklasse {r['Mehrheit']:.3f}")
print(f"Baselines: NYMEX-Regel {r['DA_NYMEX_Regel']:.3f} | Momentum {r['DA_Momentum']:.3f}")
print(f"Konfusion: TP {r['TP']} FP {r['FP']} TN {r['TN']} FN {r['FN']} | Precision {r['Precision']:.3f} Recall {r['Recall']:.3f}")
print(f"Brier-Skill {r['Brier_Skill']:.3f}")
print(f"Ersparnis gegenüber 'immer kaufen' pro Tag {r['Ersparnis_vs_kaufen']:.3f} [{r['Ersp_lo']:.3f}, {r['Ersp_hi']:.3f}]")

In [ ]:
# Zelle 7: Alle 8 Serien, S0, Test (ca. 15 Minuten). Ergebnisse werden pro Serie gesichert.
rows, RES = [], {}
t0 = time.time()
for H in SERIES:
    F = build_features(d, H)
    res = walk_forward(d, H, F, TEST, entry_lag=el, feat_lag=fl)
    RES[H] = res
    res.to_csv(f"{OUT}/db09_proba_{H}_Test_S0.csv", index=False)
    rows.append(evaluate_run(d, H, F, res, el, fl))
    print(f"{H} fertig ({time.time()-t0:.0f}s)")
ALL9 = pd.DataFrame(rows)
ALL9.to_csv(f"{OUT}/db09_ergebnisse_Test_S0.csv", index=False)

In [ ]:
# Zelle 8: Auswertung. Holm über die 8 Serien. Die 8 Serien sind NICHT unabhängig.
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
T = ALL9.copy()
T["p_holm"] = holm(T["p_nicht_ueberl"].values)
T["p_holm_NYMEX"] = holm(T["p_McNemar_NYMEX"].values)
print(T[["Serie", "n", "DA", "Wilson_lo", "Boot_lo", "Boot_hi", "p_nicht_ueberl", "p_holm",
         "DA_NYMEX_Regel", "DA_Momentum", "p_holm_NYMEX", "Brier_Skill"]].round(3).to_string(index=False))
print("\nKosten gegenüber 'immer kaufen' (Ersparnis > 0 = Modell besser):")
print(T[["Serie", "Regret_Modell", "Regret_immer_kaufen", "Ersparnis_vs_kaufen", "Ersp_lo", "Ersp_hi"]].round(3).to_string(index=False))
print("\nSerien mit Bootstrap-Untergrenze > 0.5:", int((T["Boot_lo"] > 0.5).sum()), "von", len(T))

## Interpretation (nach Lauf ausfüllen)
- Was zeigen die Ergebnisse? (Trefferquote, Bootstrap, Holm, Baselines)
- Was zeigen sie **nicht**? (Szenario S0 setzt voraus, dass zum Argus-Preis von t noch gekauft werden kann. Kein neuer Zeitraum. 8 Serien sind abhängig.)
- Empfehlung erst nach dem Holdout-Test (siehe `docs/PRAEREGISTRIERUNG_holdout_A.md`).